# Stage 1: Dynamic Dark Energy Model Selection

In [ ]:
!pip install numpy pandas matplotlib scipy emcee corner

In [ ]:
import numpy as np
import pandas as pd
import emcee
import corner
import matplotlib.pyplot as plt
from scipy.integrate import quad
import time
import urllib.request
import warnings

warnings.filterwarnings('ignore')

print("=" * 80)
print("PAPER-LEVEL JOINT ANALYSIS: Pantheon+ FULL + BAO + rd Prior + Model Selection")
print("=" * 80)

# =========================================================
# 1. DATA ACQUISITION & COVARIANCE PROCESSING
# =========================================================

# --- Pantheon+ Supernovae Data ---
print("-> Fetching Pantheon+ data dynamically from GitHub repository...")
url_sn = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
data = pd.read_csv(url_sn, sep=r'\s+')
mask = data['zHD'] > 0.01
z_sn = data['zHD'][mask].values
mu_obs = data['MU_SH0ES'][mask].values
print(f"   Number of SN Ia used (zHD > 0.01): {len(z_sn):,}")

# --- Pantheon+ Full Covariance Matrix ---
print("-> Downloading Pantheon+ Full Covariance Matrix (may take 10-15 sec)...")
url_cov = "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES_STAT%2BSYS.cov"

with urllib.request.urlopen(url_cov) as f:
    first_line = f.readline().decode('utf-8').strip()
    N = int(first_line)
print(f"   Covariance matrix size: {N} × {N}")

cov_full_flat = np.loadtxt(url_cov, skiprows=1)
cov_full = cov_full_flat.reshape(N, N)

idx = np.where(mask)[0]
cov_sn = cov_full[np.ix_(idx, idx)]
Cinv_sn = np.linalg.inv(cov_sn)
print("   Pantheon Covariance matrix successfully loaded and inverted.")

# --- SDSS DR12 BAO Consensus Data ---
print("-> Processing SDSS DR12 BAO measurements and Covariance matrix...")
z_bao = np.array([0.38, 0.51, 0.61])
bao_data = np.array([10.23, 13.36, 16.40])

cov_bao = np.array([
    [0.0400, 0.0175, 0.0120],
    [0.0175, 0.0441, 0.0215],
    [0.0120, 0.0215, 0.0576]
])
Cinv_bao = np.linalg.inv(cov_bao)
print("   BAO Covariance matrix successfully loaded and inverted.")

# =========================================================
# 2. COSMOLOGICAL MODEL DEFINITIONS
# =========================================================
c = 299792.458

def E_z(z, Om, Gamma):
    return np.sqrt(Om * (1 + z) ** 3 + (1 - Om) * np.exp(Gamma * z / (1 + z)))

def comoving_distance(z, H0, Om, Gamma):
    integral = quad(lambda zp: 1.0 / E_z(zp, Om, Gamma), 0, z)[0]
    return (c / H0) * integral

def luminosity_distance(z, H0, Om, Gamma):
    return (1 + z) * comoving_distance(z, H0, Om, Gamma)

def DV(z, H0, Om, Gamma):
    DM = comoving_distance(z, H0, Om, Gamma)
    Hz = H0 * E_z(z, Om, Gamma)
    return ((DM ** 2 * (c * z / Hz)) ** (1 / 3))

# =========================================================
# 3. OBSERVATIONAL CHI-SQUARED FUNCTIONS
# =========================================================
def pantheon_chi2(H0, Om, Gamma):
    mu_model = np.array([
        5 * np.log10(luminosity_distance(z, H0, Om, Gamma)) + 25
        for z in z_sn
    ])
    delta = mu_obs - mu_model
    ones = np.ones(len(z_sn))
    A = delta @ Cinv_sn @ delta
    B = delta @ Cinv_sn @ ones
    C = ones @ Cinv_sn @ ones
    return A - (B ** 2) / C

def bao_chi2(H0, Om, Gamma, rd):
    model = np.array([DV(z, H0, Om, Gamma) / rd for z in z_bao])
    delta = bao_data - model
    return delta.T @ Cinv_bao @ delta

def rd_prior(rd):
    rd_mean = 147.1
    rd_sigma = 0.3
    return ((rd - rd_mean) / rd_sigma) ** 2

# =========================================================
# 4. LIKELIHOOD DISTRIBUTIONS
# =========================================================
def loglike_LCDM(params):
    H0, Om, rd = params
    if not (60 < H0 < 80 and 0.2 < Om < 0.4 and 140 < rd < 155):
        return -np.inf
    Gamma = 0.0
    try:
        chi2 = pantheon_chi2(H0, Om, Gamma)
        chi2 += bao_chi2(H0, Om, Gamma, rd)
        chi2 += rd_prior(rd)
        return -0.5 * chi2
    except:
        return -np.inf

def loglike_gamma(params):
    H0, Om, Gamma, rd = params
    if not (60 < H0 < 80 and 0.2 < Om < 0.4 and -2 < Gamma < 2 and 140 < rd < 155):
        return -np.inf
    try:
        chi2 = pantheon_chi2(H0, Om, Gamma)
        chi2 += bao_chi2(H0, Om, Gamma, rd)
        chi2 += rd_prior(rd)
        return -0.5 * chi2
    except:
        return -np.inf

# =========================================================
# 5. MCMC SAMPLING MODULE
# =========================================================
def run_mcmc(loglike, ndim, initial):
    nwalkers = 40
    pos = initial + 1e-4 * np.random.randn(nwalkers, ndim)
    sampler = emcee.EnsembleSampler(nwalkers, ndim, loglike)
    sampler.run_mcmc(pos, 2000, progress=True)
    samples = sampler.get_chain(discard=500, flat=True)
    return samples

# =========================================================
# 6. EXECUTION PIPELINE
# =========================================================
print("\n-> TEST 1: Running MCMC for Standard ΛCDM Model...")
samples_LCDM = run_mcmc(loglike_LCDM, 3, np.array([70, 0.3, 147]))

print("\n-> TEST 2: Running MCMC for Dynamic Dark Energy (Gamma) Model...")
samples_GAMMA = run_mcmc(loglike_gamma, 4, np.array([70, 0.3, 0.0, 147]))

# =========================================================
# 7. MODEL SELECTION STATISTICS (AIC / BIC)
# =========================================================
def compute_stats(samples, model_type):
    best = np.mean(samples, axis=0)
    if model_type == "LCDM":
        H0, Om, rd = best
        Gamma = 0
        k = 3
    else:
        H0, Om, Gamma, rd = best
        k = 4
    chi2 = pantheon_chi2(H0, Om, Gamma)
    chi2 += bao_chi2(H0, Om, Gamma, rd)
    chi2 += rd_prior(rd)
    N = len(z_sn) + len(z_bao)
    AIC = chi2 + 2 * k
    BIC = chi2 + k * np.log(N)
    return chi2, AIC, BIC

chi2_L, AIC_L, BIC_L = compute_stats(samples_LCDM, "LCDM")
chi2_G, AIC_G, BIC_G = compute_stats(samples_GAMMA, "GAMMA")

# =========================================================
# 8. FINAL RESULTS & CONCLUSIONS
# =========================================================
print("\n" + "=" * 50)
print("MODEL SELECTION STATISTICS")
print("=" * 50)

print(f"ΛCDM Model  -> Chi2={chi2_L:.2f}, AIC={AIC_L:.2f}, BIC={BIC_L:.2f}")
print(f"Gamma Model -> Chi2={chi2_G:.2f}, AIC={AIC_G:.2f}, BIC={BIC_G:.2f}")

print("\nDELTA SCORES (Gamma - ΛCDM):")
print(f"ΔAIC = {AIC_G - AIC_L:.2f}")
print(f"ΔBIC = {BIC_G - BIC_L:.2f}")

print("\nCONCLUSION:")
if (BIC_G - BIC_L) < -10:
    print(">>> DYNAMIC DARK ENERGY (GAMMA) STRONGLY PREFERRED BY DATA.")
elif (BIC_G - BIC_L) < -2:
    print(">>> WEAK EVIDENCE FOR DYNAMIC DARK ENERGY (GAMMA).")
else:
    print(">>> STANDARD ΛCDM MODEL IS STATISTICALLY PREFERRED (Occam's Razor).")

# =========================================================
# 9. POSTERIOR VISUALIZATION
# =========================================================
fig = corner.corner(
    samples_GAMMA,
    labels=[r"$H_0$", r"$\Omega_m$", r"$\Gamma$", r"$r_d$"],
    show_titles=True,
    title_kwargs={"fontsize": 12}
)
plt.show()